# World Taxation Systems Analysis
**Analyst:** Juan Xavier Gomez Illingworth

**NB:** Parts of the code were constructed with the assistance of Gen AI.


In [10]:
# Import libraries for scraping, cleaning, and visualization
import requests
from bs4 import BeautifulSoup
import pandas as pd
import altair as alt
from vega_datasets import data
import time
import json


In [11]:
# Scrape taxation systems from the International taxation page
import re

url1 = "https://en.wikipedia.org/wiki/International_taxation"
request_headers = {
    'User-Agent': 'Educational Project (contact: student@university.edu)',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'
}

response1 = requests.get(url1, headers=request_headers)
time.sleep(1)

soup1 = BeautifulSoup(response1.content, 'html.parser')

tables1 = soup1.find_all('table', {'class': 'wikitable'})

def extract_taxation_system(notes, taxes_foreign_citizens):
    """Extract the taxation system type from Notes and tax columns"""
    if pd.isna(notes) or notes == '':
        return 'Unknown'

    notes_lower = notes.lower()

    if ('citizenship-based' in notes_lower or 'citizen-based' in notes_lower or
        (taxes_foreign_citizens == 'Yes' and 'citizenship' in notes_lower)):
        return 'Citizenship-based'
    elif 'no personal income tax' in notes_lower:
        return 'No income tax'
    elif 'territorial taxation' in notes_lower:
        return 'Territorial'
    elif 'residential taxation' in notes_lower or 'residence-based' in notes_lower:
        return 'Residential'
    else:
        match = re.search(r'(\w+(?:\s+\w+)*)\s+taxation', notes_lower)
        if match:
            return match.group(1).title()
        return 'Other'

taxation_systems = []
if len(tables1) > 0:
    table = tables1[0]
    rows = table.find_all('tr')
    header_row1 = rows[0]
    header_row2 = rows[1] if len(rows) > 1 else None
    main_headers = header_row1.find_all('th')

    columns = []
    for th in main_headers:
        text = th.get_text(strip=True)
        colspan = int(th.get('colspan', 1))

        if colspan > 1 and header_row2:
            sub_headers = header_row2.find_all(['th', 'td'])
            for j in range(colspan):
                if len(sub_headers) > len(columns):
                    sub_text = sub_headers[len(columns)].get_text(strip=True)
                    columns.append(f"{text} {sub_text}".strip())
        else:
            columns.append(text)

    for row in rows[2:]:
        cells = row.find_all(['td', 'th'])
        if len(cells) >= 4:
            country = cells[0].get_text(strip=True)
            if country and len(country) > 2 and country.lower() not in ['country', 'jurisdiction', 'territory']:
                notes = cells[4].get_text(strip=True) if len(cells) > 4 else ''
                taxes_foreign_citizens = cells[3].get_text(strip=True) if len(cells) > 3 else ''

                row_data = {
                    'Country': country,
                    'Taxes local income': cells[1].get_text(strip=True) if len(cells) > 1 else '',
                    'Taxes foreign income of residents': cells[2].get_text(strip=True) if len(cells) > 2 else '',
                    'Taxes foreign income of non-resident citizens': taxes_foreign_citizens,
                    'Notes': notes,
                    'Taxation System': extract_taxation_system(notes, taxes_foreign_citizens)
                }
                taxation_systems.append(row_data)

df_systems = pd.DataFrame(taxation_systems)
df_systems[['Country', 'Taxation System']].head()
df_systems['Taxation System'].value_counts()


Taxation System
Residential          169
Territorial           45
No income tax         19
Citizenship-based      5
Name: count, dtype: int64

In [12]:
# Scrape tax rates by country from Wikipedia and strip bracketed references
import re

url2 = "https://en.wikipedia.org/wiki/List_of_countries_by_tax_rates"

response2 = requests.get(url2, headers=request_headers)
time.sleep(1)

soup2 = BeautifulSoup(response2.content, 'html.parser')


def remove_brackets(text):
    return re.sub(r"\s*\[[^\]]*\]", "", text or "").strip()


tax_rates = []
table = soup2.find('table', {'class': 'wikitable'})

if table:
    rows = table.find_all('tr')
    header_row1 = rows[0].find_all(['th', 'td'])
    header_row2 = rows[1].find_all(['th', 'td'])

    columns = []
    sub_idx = 0

    for th in header_row1:
        text = remove_brackets(th.get_text(strip=True))
        colspan = int(th.get('colspan', 1))
        rowspan = int(th.get('rowspan', 1))

        if colspan > 1:
            for j in range(colspan):
                if sub_idx < len(header_row2):
                    sub_text = remove_brackets(header_row2[sub_idx].get_text(strip=True))
                    columns.append(f"{text} {sub_text}".strip())
                    sub_idx += 1
        else:
            columns.append(text)

    for row in rows[2:]:
        cells = row.find_all(['td', 'th'])
        if len(cells) >= len(columns):
            country = remove_brackets(cells[0].get_text(strip=True))
            if country and len(country) > 2 and country.lower() not in ['country', 'jurisdiction', 'lowest', 'highest']:
                row_data = {}
                for i, col_name in enumerate(columns):
                    cell_text = remove_brackets(cells[i].get_text(strip=True)) if i < len(cells) else ''
                    row_data[col_name] = cell_text

                tax_rates.append(row_data)

df_tax_rates = pd.DataFrame(tax_rates)
df_tax_rates.head(20)


,Tax jurisdiction,Corporate,Individual income Lowest,Individual income Highest,Capital gains,Wealth,Property,Inheritance/Estate,VAT or GSTorSales,Further reading
0,Afghanistan,20%,0%,20%,,,,,"0%(however, in Taliban run areas pre-Taliban r...",Taxation in Afghanistan
1,Albania,15%,0%,23%,15%,,,,20%(standard)6%(tourism services),Taxation in Albania
2,Algeria,19–26%,0%,35%,15%(resident)20%(non-resident),,,,19%(standard)9%(basic items),Taxation in Algeria
3,American Samoa,34%,4%,6%,,,0%,,0%,Taxation in American Samoa
4,Andorra,10%,0%,10%,,,,,"4.5%(standard)9.5%(banking services)2.5%, 1% o...",Taxation in Andorra
5,Angola,30%,0%,17%,10%,,,,14%,Taxation in Angola
6,Argentina,35%(residents)15%(non-residents),9%,35%,15%,,,,21%,Taxation in Argentina
7,Armenia,18%,22%,22%,10–20%,,,,20%,Taxation in Armenia
8,Aruba,25%,7%,58.95%,,,,,1.5%(turnover tax),Taxation in Aruba
9,Australia,30%(standard)25%(base entity),0%,45%,0–45%,No,,0%,10%(standard)0%(essential items),Taxation in Australia


In [13]:
# Clean and merge taxation systems with tax rates

def clean_country_name(name):
    name = name.split('[')[0].strip()
    name = name.split('(')[0].strip()
    return name

if 'Country' in df_systems.columns:
    df_systems['Country_clean'] = df_systems['Country'].apply(clean_country_name)
else:
    print("Warning: 'Country' column not found in taxation systems dataframe")

if 'Tax jurisdiction' in df_tax_rates.columns:
    df_tax_rates['Country_clean'] = df_tax_rates['Tax jurisdiction'].apply(clean_country_name)
else:
    print("Warning: Tax jurisdiction column not found in tax rates dataframe")
    print(f"Available columns: {list(df_tax_rates.columns)}")

df_merged = pd.merge(
    df_tax_rates,
    df_systems,
    on='Country_clean',
    how='outer'
)

df_merged.head(20)


,Tax jurisdiction,Corporate,Individual income Lowest,Individual income Highest,Capital gains,Wealth,Property,Inheritance/Estate,VAT or GSTorSales,Further reading,Country_clean,Country,Taxes local income,Taxes foreign income of residents,Taxes foreign income of non-resident citizens,Notes,Taxation System
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Abkhazia,Abkhazia,Yes,Yes,No,Residence-based taxation.[46],Residential
1,Afghanistan,20%,0%,20%,,,,,"0%(however, in Taliban run areas pre-Taliban r...",Taxation in Afghanistan,Afghanistan,Afghanistan,Yes,Yes,No,Residence-based taxation.[6],Residential
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Akrotiri and Dhekelia,Akrotiri and Dhekelia,Yes,Yes,No,Residence-based taxation.[47],Residential
3,Albania,15%,0%,23%,15%,,,,20%(standard)6%(tourism services),Taxation in Albania,Albania,Albania,Yes,Yes,No,Residence-based taxation.[6],Residential
4,Algeria,19–26%,0%,35%,15%(resident)20%(non-resident),,,,19%(standard)9%(basic items),Taxation in Algeria,Algeria,Algeria,Yes,Yes,No,Residence-based taxation.[6],Residential
5,American Samoa,34%,4%,6%,,,0%,,0%,Taxation in American Samoa,American Samoa,American Samoa,Yes,Yes,No,Residence-based taxation.[48],Residential
6,Andorra,10%,0%,10%,,,,,"4.5%(standard)9.5%(banking services)2.5%, 1% o...",Taxation in Andorra,Andorra,Andorra,Yes,Yes,No,Residence-based taxation.[49],Residential
7,Angola,30%,0%,17%,10%,,,,14%,Taxation in Angola,Angola,Angola,Yes,No,No,Territorial taxation.[6],Territorial
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Anguilla,Anguilla,Yes,No,No,Territorial taxation.[21],Territorial
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Antigua and Barbuda,Antigua and Barbuda,No,No,No,No personal income tax.[4][5],No income tax


In [14]:
# Prepare merged data for visualization and country name mapping
from vega_datasets import data as vega_data

countries = alt.topo_feature(vega_data.world_110m.url, 'countries')

df_viz = df_merged.copy()

country_mapping = {
    'United States': 'United States of America',
    'United Kingdom': 'United Kingdom',
    'Russia': 'Russia',
    'South Korea': 'South Korea',
    'North Korea': 'North Korea',
    'DR Congo': 'Dem. Rep. Congo',
    'Congo': 'Congo',
    'Tanzania': 'Tanzania',
    'Ivory Coast': "Côte d'Ivoire",
    'Czech Republic': 'Czechia',
    'Timor-Leste': 'Timor-Leste',
    'Eswatini': 'eSwatini',
    'Serbia': 'Serbia',
    'Bosnia and Herzegovina': 'Bosnia and Herz.',
    'Dominican Republic': 'Dominican Rep.',
    'Central African Republic': 'Central African Rep.',
    'South Sudan': 'S. Sudan',
    'Solomon Islands': 'Solomon Is.',
    'Equatorial Guinea': 'Eq. Guinea',
    'Guinea-Bissau': 'Guinea-Bissau',
    'W. Sahara': 'W. Sahara'
}

df_viz['name'] = df_viz['Country_clean'].replace(country_mapping)

if 'Taxation System' in df_viz.columns:
    df_viz['Taxation System'] = df_viz['Taxation System'].fillna('Unknown')
else:
    df_viz['Taxation System'] = 'Unknown'

df_viz = df_viz.rename(columns={
    'Individual income Lowest': 'Income_Tax_Min',
    'Individual income Highest': 'Income_Tax_Max',
    'VAT or GSTorSales': 'VAT_GST_Sales'
})

viz_columns = ['name', 'Taxation System', 'Corporate', 'Income_Tax_Min', 'Income_Tax_Max', 'VAT_GST_Sales']
df_viz_clean = df_viz[viz_columns].copy()

df_viz_clean.head()
df_viz_clean['Taxation System'].value_counts()


Taxation System
Residential          169
Territorial           45
No income tax         19
Unknown                6
Citizenship-based      5
Name: count, dtype: int64

In [15]:
# Build an interactive rotating globe of taxation systems and tax rates
alt.data_transformers.enable('default')

world_url = 'https://cdn.jsdelivr.net/npm/world-atlas@2/countries-110m.json'
world_countries = alt.topo_feature(world_url, 'countries')

rotate0 = alt.param(name='rotate0', value=0, bind=alt.binding_range(name='Rotate Horizontally', min=-180, max=180, step=1))
rotate1 = alt.param(name='rotate1', value=0, bind=alt.binding_range(name='Rotate Vertically', min=-90, max=90, step=1))

sphere = alt.Chart(alt.sphere()).mark_geoshape(fill='aliceblue')

color_scheme = {
    'Citizenship-based': '#4169E1',
    'Residential': '#FF9F80',
    'Territorial': '#32CD32',
    'No income tax': '#FFD700',
    'Unknown': '#D3D3D3',
    'Other': '#9370DB'
}

countries_layer = alt.Chart(world_countries).mark_geoshape(
    stroke='white',
    strokeWidth=0.5
).transform_lookup(
    lookup='properties.name',
    from_=alt.LookupData(
        df_viz_clean,
        'name',
        ['name', 'Taxation System', 'Corporate', 'Income_Tax_Min', 'Income_Tax_Max', 'VAT_GST_Sales']
    )
).encode(
    color=alt.condition(
        alt.datum['Taxation System'] != None,
        alt.Color('Taxation System:N',
                  scale=alt.Scale(domain=list(color_scheme.keys()),
                                 range=list(color_scheme.values())),
                  legend=alt.Legend(title='Taxation System')),
        alt.value('mintcream')
    ),
    tooltip=[
        alt.Tooltip('properties.name:N', title='Country'),
        alt.Tooltip('Taxation System:N', title='Taxation System'),
        alt.Tooltip('Corporate:N', title='Corporate Tax'),
        alt.Tooltip('Income_Tax_Min:N', title='Income Tax (Min)'),
        alt.Tooltip('Income_Tax_Max:N', title='Income Tax (Max)'),
        alt.Tooltip('VAT_GST_Sales:N', title='VAT/GST/Sales Tax')
    ]
)

chart = alt.layer(sphere, countries_layer).add_params(
    rotate0, rotate1
).project(
    type='orthographic',
    rotate={'expr': '[rotate0, rotate1, 0]'}
).properties(
    width=600,
    height=600,
    title={
        'text': 'Interactive Globe: World Taxation Systems and Tax Rates',
        'subtitle': [
            'Source: Wikipedia (International Taxation, List of Countries by Tax Rates).',
            'World URL: https://cdn.jsdelivr.net/npm/world-atlas@2/countries-110m.json'
        ],
        'anchor': 'start'
    }
).configure_view(
    stroke=None
)

chart


alt.LayerChart(...)

In [16]:
# Save the visualization to disk
output_path = r'c:\Users\juanx\Documents\GitHub\juanxgi83.github.io\graphs\world_taxation_map.json'
chart.save(output_path)
print(f"Chart saved to: {output_path}")


Chart saved to: c:\Users\juanx\Documents\GitHub\juanxgi83.github.io\graphs\world_taxation_map.json
